# ðŸš¨ Crisis Negotiator â€” GRPO Training Notebook

**Multi-Agent De-escalation RL Environment** | OpenEnv Hackathon Submission

This notebook trains a Qwen2.5-7B model to act as an FBI-style crisis negotiator using:
- **GRPO** (Group Relative Policy Optimization) from HuggingFace TRL
- **Unsloth** 4-bit quantization for free Colab T4 GPU
- **Adaptive curriculum** that auto-escalates difficulty (easy â†’ medium â†’ hard)
- **Self-improvement**: failure-adaptive scenario mutation generates harder variants of failed episodes
- **Multi-component reward**: outcome, FBI technique usage, token efficiency (Mercor), safety oversight
- **Expert-in-the-loop** (Snorkel AI): rotating expert feedback with changing requirements

**Requirements**: Colab T4 GPU (free tier) or local GPU with â‰¥16GB VRAM

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl openenv-core peft accelerate datasets sentence-transformers
!pip install -q openai fastapi uvicorn pydantic requests numpy matplotlib

In [ ]:
# Cell 2: Clone repo (skip if already local)
import os
if not os.path.exists('server'):
    !git clone https://github.com/Dinesh052/Test.git crisis-negotiator
    %cd crisis-negotiator

In [ ]:
# Cell 3: Load model with Unsloth 4-bit LoRA
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length=4096,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: {model.config._name_or_path}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Cell 4: Initialize environment + curriculum + self-improvement
import sys, json, re
sys.path.insert(0, '.')

from server.environment import CrisisNegotiatorEnvironment
from server.scenario_generator import AdaptiveCurriculum, FailureAdaptiveGenerator
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
curriculum = AdaptiveCurriculum(window=10, threshold=0.7)
failure_gen = FailureAdaptiveGenerator(failure_threshold=0.4)

# Verify environment works
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
obs = env.step(NegotiatorAction(
    action_type='emotional_label',
    content="It sounds like you're feeling overwhelmed.",
    reasoning='test', target='hostage_taker'
))
print(f"Environment OK: step reward={obs.reward:.3f}")
print(f"Curriculum: {curriculum.stats}")
print(f"Self-improvement: {failure_gen.stats}")

In [ ]:
# Cell 5: Define system prompt and helpers

SYSTEM_PROMPT = """You are an expert FBI-trained crisis negotiator. De-escalate the situation using:
- Emotional Labeling: "It sounds like you're feeling..."
- Mirroring: Repeat their last words
- Open Questions: "Tell me more about..."
- Demand Acknowledgment: Acknowledge without committing
- Stay calm. Never threaten. Never dismiss demands.

Respond with ONE JSON object:
{"action_type": "emotional_label|mirror|open_question|acknowledge_demand|speak|offer_concession|buy_time", "content": "your words", "reasoning": "your strategy", "target": "hostage_taker"}"""


def build_prompt(obs) -> str:
    parts = [f"Scenario: {obs.scenario_brief}"]
    parts.append(f"Step {obs.step}, Time left: {obs.time_remaining}")
    if obs.dialogue_history:
        for e in obs.dialogue_history[-4:]:
            parts.append(f"{e['speaker'].upper()}: {e['content']}")
    if obs.stated_demands:
        parts.append(f"Demands: {[d['text'] for d in obs.stated_demands]}")
    if obs.commander_patience != 'patient':
        parts.append(f"Commander: {obs.commander_patience}")
    return '\n'.join(parts)


def parse_action(text: str) -> dict:
    text = re.sub(r'```(?:json)?\s*', '', text.strip())
    text = re.sub(r'```\s*$', '', text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{[^{}]*\}', text)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    return {'action_type': 'speak', 'content': text[:200], 'reasoning': '', 'target': 'hostage_taker'}

print("Helpers defined.")

In [ ]:
# Cell 6: GRPO reward function with self-improvement

def run_episode(model_output: str, scenario_seed: int) -> float:
    """Run one episode and return terminal grader score."""
    scenario = curriculum.get_scenario(seed=scenario_seed)
    obs = env.reset(task_id='generate:' + scenario['difficulty'], seed=scenario_seed)

    action_dict = parse_action(model_output)
    try:
        action = NegotiatorAction(**action_dict)
    except Exception:
        action = NegotiatorAction(action_type='speak', content=model_output[:200],
                                 reasoning='', target='hostage_taker')

    obs = env.step(action)

    # Continue with heuristic policy for remaining steps
    for _ in range(20):
        if obs.done:
            break
        action = NegotiatorAction(
            action_type='emotional_label',
            content="I hear you. Tell me more about how you're feeling.",
            reasoning='maintain rapport', target='hostage_taker',
        )
        obs = env.step(action)

    # Use terminal grader score (NOT max of step rewards)
    final_reward = obs.reward if obs.done else max(0.01, min(0.99, 0.3))

    # Self-improvement: curriculum + failure-adaptive mutation
    curriculum.record(scenario.get('difficulty', 'medium'), final_reward)
    failure_gen.on_episode_end(scenario, final_reward, seed=scenario_seed)
    return final_reward


from typing import List

def reward_fn(completions: List[str], **kwargs) -> List[float]:
    """GRPO reward function â€” runs each completion through the environment."""
    rewards = []
    for i, completion in enumerate(completions):
        try:
            r = run_episode(completion, scenario_seed=i)
        except Exception:
            r = 0.01
        rewards.append(r)
    return rewards

print("Reward function defined.")

In [ ]:
# Cell 7: GRPO Training â€” connects to live environment, logs rewards per step

from unsloth import PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import json, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# â”€â”€ Dataset: opening lines covering all difficulty tiers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
OPENING_LINES = [
    # Easy
    "SUSPECT: Nobody move! I just want to see my kids! [easy]",
    "SUSPECT: I don't want to hurt anyone. I just need someone to listen. [easy]",
    "SUSPECT: I've got nothing left. They took everything from me. [easy]",
    "SUSPECT: Please, I just want this to end peacefully. [easy]",
    # Medium
    "SUSPECT: I have demands and I'm not leaving until they're met. [medium]",
    "SUSPECT: You think you can trick me? I know what you're doing. [medium]",
    "SUSPECT: Back off! I can do this all day. [medium]",
    "SUSPECT: My cause is righteous. You wouldn't understand. [medium]",
    "SUSPECT: Every minute you stall, things get worse in here. [medium]",
    # Hard
    "SUSPECT: I have 8 hostages and I want safe passage out of the country. [hard]",
    "SUSPECT: You have 10 minutes before I start making decisions for you. [hard]",
    "SUSPECT: Tell your snipers to stand down or this ends badly for everyone. [hard]",
]

dataset = Dataset.from_list([
    {"prompt": f"{SYSTEM_PROMPT}\n\nSituation: Active crisis negotiation.\n{line}\n\nNEGOTIATOR:"}
    for line in OPENING_LINES * 5  # 60 examples
])

# â”€â”€ GRPOConfig (T4 16GB optimised) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
training_args = GRPOConfig(
    output_dir="crisis-negotiator-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_new_tokens=200,
    max_prompt_length=512,
    learning_rate=5e-5,
    logging_steps=5,
    save_steps=100,
    warmup_ratio=0.05,
    report_to="none",
    remove_unused_columns=False,
)

# â”€â”€ Reward logging wrapper â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
reward_log = []

def logged_reward_fn(completions, **kwargs):
    rewards = reward_fn(completions, **kwargs)
    reward_log.extend([{"step": len(reward_log) + i, "reward": float(r)} for i, r in enumerate(rewards)])
    if len(reward_log) % 20 == 0:
        recent = [e["reward"] for e in reward_log[-20:]]
        print(f"  step {len(reward_log):4d} | mean reward: {sum(recent)/len(recent):.3f}")
    return rewards

# â”€â”€ Train â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=logged_reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print(f"Starting GRPO training | {len(dataset)} prompts Ã— {training_args.num_generations} generations")
trainer.train()
print("\nTraining complete!")

# â”€â”€ Save reward log â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
with open("reward_log.json", "w") as f:
    json.dump(reward_log, f, indent=2)
print(f"Saved reward_log.json ({len(reward_log)} entries)")

# â”€â”€ Plot and save reward curve â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
rewards_list = [e["reward"] for e in reward_log]
window = max(1, len(rewards_list) // 10)
rolling = np.convolve(rewards_list, np.ones(window) / window, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(rewards_list, alpha=0.3, color='steelblue', label='Per-completion reward')
ax1.plot(range(window - 1, len(rewards_list)), rolling, color='steelblue', linewidth=2, label=f'Rolling avg (w={window})')
ax1.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Success threshold (0.5)')
ax1.set_xlabel('Training step')
ax1.set_ylabel('Reward')
ax1.set_title('GRPO Reward â€” Crisis Negotiator (Colab T4)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Success rate over time in 10 equal windows
chunk = max(1, len(rewards_list) // 10)
success_rates = [sum(1 for r in rewards_list[i:i+chunk] if r > 0.5) / chunk
                 for i in range(0, len(rewards_list) - chunk + 1, chunk)]
ax2.bar(range(len(success_rates)), success_rates, color='steelblue', alpha=0.7)
ax2.axhline(success_rates[0] if success_rates else 0, color='gray', linestyle='--', linewidth=0.8, label='Baseline')
ax2.set_xlabel('Training window (each = 10% of steps)')
ax2.set_ylabel('Success rate (reward > 0.5)')
ax2.set_title('Success Rate Progression')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved reward_curve.png")

In [ ]:
# Cell 8: Display reward curve (generated by training cell above)
from IPython.display import Image
Image('reward_curve.png')

In [ ]:
# Cell 9: Summary stats

# reward_log is populated in-memory by the training cell above
data = reward_log
rewards = [d['reward'] for d in data]
n = len(rewards)
tenth = max(1, n // 10)
first = rewards[:tenth]
last = rewards[-tenth:]

print(f'Total completions logged: {n}')
print(f'First 10% avg reward:     {sum(first)/len(first):.3f}')
print(f'Last 10% avg reward:      {sum(last)/len(last):.3f}')
print(f'Improvement:              {sum(last)/len(last) - sum(first)/len(first):+.3f}')
print(f'Overall success rate:     {sum(1 for r in rewards if r > 0.5)/n:.0%}')
print(f'Best reward:              {max(rewards):.3f}')
print(f'Worst reward:             {min(rewards):.3f}')

In [ ]:
# Cell 10: Download artifacts
from google.colab import files
files.download('reward_curve.png')
files.download('reward_log.json')